# Week 6 — Housing, Safety, Population, and School-District Geography

This notebook tests four cumulative feature sets while keeping the chronological split, preprocessing, and model settings fixed. Section 6 introduces the exact measurements added at each stage under separate subtitles. Airport noise and environmental-burden measurements remain outside the current scope.

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import urllib.request

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, TargetEncoder
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42

## 1. Data sources

Property transactions come from the same confidential CRMLS monthly files and cleaned chronological splits used in Notebooks 02–04.

School-district boundaries come from the California Department of Education's **California School District Areas 2024–25** layer, published through the California Open Data Portal. The official layer combines elementary, high, and unified district polygons. Elementary and high districts can overlap because they serve different grade ranges within the same territory.

- [Dataset description and downloads](https://data.ca.gov/@california-department-of-education/california-school-district-areas-2024-25)
- [Official ArcGIS feature layer and field definitions](https://services3.arcgis.com/fdvHcZVgB2QSRNkL/arcgis/rest/services/SchoolDistrictAreas2425/FeatureServer/0)

Only `CDCode`, `DistrictName`, `DistrictType`, and grade-range fields are retained. Student demographic attributes are not used.

The population-density feature uses **2024 ACS five-year estimates** joined to **2024 Census tract boundaries**. Density is used instead of total population because tract land areas differ greatly. ACS values are estimates for 2020–2024, not exact 2026 counts.

- [2024 ACS five-year Summary File](https://www.census.gov/programs-surveys/acs/data/summary-file.2024.html)
- [2024 TIGER/Line tract boundaries](https://www2.census.gov/geo/tiger/TIGER2024/TRACT/)

County violent- and property-crime counts come from the California Department of Justice **Crimes and Clearances (including Arson)** annual summary. The DOJ data is reported by law-enforcement jurisdictions under the FBI Uniform Crime Reporting program. Violent crime combines homicide, rape, robbery, and aggravated assault; property crime combines burglary, larceny-theft, and motor-vehicle theft. Jurisdiction counts are summed within each county and divided by the corresponding ACS county population.

- [California DOJ Crimes and Clearances dataset](https://data-openjustice.doj.ca.gov/data/crimes-and-clearances-including-arson)
- [California DOJ data characteristics and limitations](https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2026-07/Crimes-Clear-Arson%20Context_06222026.pdf)
- [Census county code reference](https://www2.census.gov/geo/docs/reference/codes2020/)

A conservative two-calendar-year lag is used: 2024 sales receive 2022 crime data, 2025 sales receive 2023 data, and 2026 sales receive 2024 data. This prevents same-year crime outcomes from being used to predict earlier sales.

In [ ]:
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
EXTERNAL_DIR = REPO_ROOT / "data" / "external"
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
DISTRICT_DATASET_PAGE = (
    "https://data.ca.gov/@california-department-of-education/"
    "california-school-district-areas-2024-25"
)
DISTRICT_SERVICE = (
    "https://services3.arcgis.com/fdvHcZVgB2QSRNkL/arcgis/rest/services/"
    "SchoolDistrictAreas2425/FeatureServer/0"
)
DISTRICT_DOWNLOAD_URL = (
    DISTRICT_SERVICE + "/query?where=1%3D1"
    + "&outFields=Year%2CFedID%2CCDCode%2CCDSCode%2CCountyName%2C"
      "DistrictName%2CDistrictType%2CGradeLow%2CGradeHigh"
    + "&returnGeometry=true&outSR=4326&f=geojson"
)
DISTRICT_PATH = EXTERNAL_DIR / "ca_school_district_areas_2024_25.geojson"

ACS_YEAR = 2024
SAFETY_YEARS = (2022, 2023, 2024)

# Build the official ACS B01003 population-file URL for one year.
def acs_population_url(year):
    return (
        f"https://www2.census.gov/programs-surveys/acs/summary_file/{year}/"
        f"table-based-SF/data/5YRData/acsdt5y{year}-b01003.dat"
    )

ACS_POPULATION_URLS = {year: acs_population_url(year) for year in SAFETY_YEARS}
ACS_POPULATION_PATHS = {
    year: EXTERNAL_DIR / f"acsdt5y{year}-b01003.dat"
    for year in SAFETY_YEARS
}
ACS_POPULATION_URL = ACS_POPULATION_URLS[ACS_YEAR]
ACS_POPULATION_PATH = ACS_POPULATION_PATHS[ACS_YEAR]
TRACT_URL = (
    f"https://www2.census.gov/geo/tiger/TIGER{ACS_YEAR}/TRACT/"
    f"tl_{ACS_YEAR}_06_tract.zip"
)
TRACT_PATH = EXTERNAL_DIR / f"tl_{ACS_YEAR}_06_tract.zip"

DOJ_CRIME_DATASET_PAGE = (
    "https://data-openjustice.doj.ca.gov/data/"
    "crimes-and-clearances-including-arson"
)
DOJ_CRIME_URL = (
    "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2026-07/"
    "Crimes_and_Clearances_with_Arson-1985-2025.csv"
)
DOJ_CRIME_PATH = EXTERNAL_DIR / "ca_doj_crimes_clearances_1985_2025.csv"
COUNTY_REFERENCE_URL = (
    "https://www2.census.gov/geo/docs/reference/codes2020/"
    "national_county2020.txt"
)
COUNTY_REFERENCE_PATH = EXTERNAL_DIR / "national_county2020.txt"

In [ ]:
if not DISTRICT_PATH.exists():
    print("Downloading official school-district boundaries...")
    urllib.request.urlretrieve(DISTRICT_DOWNLOAD_URL, DISTRICT_PATH)

district_sha256 = hashlib.sha256(DISTRICT_PATH.read_bytes()).hexdigest()
district_columns = [
    "Year", "CDCode", "DistrictName", "DistrictType",
    "GradeLow", "GradeHigh", "geometry",
]
districts = gpd.read_file(DISTRICT_PATH)[district_columns].copy()
assert str(districts.crs).upper() == "EPSG:4326"
assert districts["Year"].dropna().eq("2024-25").all()
assert set(districts["DistrictType"].dropna()) == {"Elementary", "High", "Unified"}

invalid_geometry_count = int((~districts.geometry.is_valid).sum())
if invalid_geometry_count:
    districts.geometry = districts.geometry.make_valid()
assert districts.geometry.is_valid.all() and ~districts.geometry.is_empty.any()
districts["CDCode"] = districts["CDCode"].astype("string").str.zfill(7)

In [ ]:
district_summary = districts.groupby("DistrictType").agg(
    Districts=("CDCode", "nunique"),
    Grade_low=("GradeLow", "min"),
    Grade_high=("GradeHigh", "max"),
)
display(district_summary)
print(f"Boundary polygons: {len(districts):,}")
print(f"Invalid geometries repaired: {invalid_geometry_count}")
print(f"Cached file SHA-256: {district_sha256}")

### California school-district boundary map

Separate panels keep the overlapping elementary, high, and unified district systems readable. All panels use the same geographic extent.

In [ ]:
district_types = ["Elementary", "High", "Unified"]
district_colors = ["#4C78A8", "#F58518", "#54A24B"]
min_x, min_y, max_x, max_y = districts.total_bounds
fig, axes = plt.subplots(1, 3, figsize=(15, 8))

for axis, district_type, color in zip(axes, district_types, district_colors):
    districts.loc[districts["DistrictType"].eq(district_type)].plot(
        ax=axis, color=color, edgecolor="white", linewidth=0.2
    )
    axis.set(
        title=f"{district_type} districts",
        xlim=(min_x, max_x), ylim=(min_y, max_y),
    )
    axis.set_axis_off()

fig.suptitle("California School District Areas 2024–25", fontsize=16)
fig.tight_layout()
plt.show()

### Population-density feature preparation

Population density remains a model feature and supports population-based crime rates, but a separate statewide population map is not needed for the Core ideas comparison. Density is calculated as ACS population divided by Census-tract land area.

In [ ]:
downloads = [(ACS_POPULATION_URL, ACS_POPULATION_PATH), (TRACT_URL, TRACT_PATH)]
for url, path in downloads:
    if not path.exists():
        print(f"Downloading {path.name}...")
        urllib.request.urlretrieve(url, path)

acs = pd.read_csv(ACS_POPULATION_PATH, sep="|", dtype={"GEO_ID": "string"})
acs = acs.loc[acs["GEO_ID"].str.startswith("1400000US06")].copy()
acs["GEOID"] = acs["GEO_ID"].str.removeprefix("1400000US")
acs["Population"] = pd.to_numeric(acs["B01003_E001"], errors="coerce")
acs = acs[["GEOID", "Population"]]

tracts = gpd.read_file(f"zip://{TRACT_PATH}")[["GEOID", "ALAND", "geometry"]]
tracts = tracts.to_crs("EPSG:3310")
population_map = tracts.merge(acs, on="GEOID", how="left", validate="one_to_one")
population_map["LandAreaSqMi"] = population_map["ALAND"] / 2_589_988.11
population_map["PopulationDensity"] = (
    population_map["Population"] / population_map["LandAreaSqMi"]
)
assert population_map["Population"].notna().mean() > 0.99

### County safety measurements

In the California DOJ data:

- **Violent crime:** homicide, rape, robbery, and aggravated assault.
- **Property crime:** burglary, larceny-theft, and motor-vehicle theft.

The model keeps these as separate incidents-per-1,000-residents rates. County rates provide statewide coverage but cannot describe neighborhood-level differences.

In [ ]:
safety_downloads = [
    (DOJ_CRIME_URL, DOJ_CRIME_PATH),
    (COUNTY_REFERENCE_URL, COUNTY_REFERENCE_PATH),
] + [
    (ACS_POPULATION_URLS[year], ACS_POPULATION_PATHS[year])
    for year in SAFETY_YEARS
]
for url, path in safety_downloads:
    if not path.exists():
        print(f"Downloading {path.name}...")
        urllib.request.urlretrieve(url, path)

county_reference = pd.read_csv(
    COUNTY_REFERENCE_PATH, sep="|", dtype="string"
).loc[lambda data: data["STATE"].eq("CA")].copy()
county_reference["GEOID"] = (
    county_reference["STATEFP"] + county_reference["COUNTYFP"]
)
county_reference["CountyKey"] = (
    county_reference["COUNTYNAME"].str.removesuffix(" County").str.casefold()
)

county_population_frames = []
for year in SAFETY_YEARS:
    population = pd.read_csv(
        ACS_POPULATION_PATHS[year], sep="|", dtype={"GEO_ID": "string"}
    )
    population = population.loc[
        population["GEO_ID"].str.startswith("0500000US06")
    ].copy()
    population["GEOID"] = population["GEO_ID"].str.removeprefix("0500000US")
    population["Population"] = pd.to_numeric(
        population["B01003_E001"], errors="coerce"
    )
    population["SafetyDataYear"] = year
    county_population_frames.append(
        population[["GEOID", "SafetyDataYear", "Population"]]
    )

county_population = pd.concat(county_population_frames, ignore_index=True).merge(
    county_reference[["GEOID", "CountyKey", "COUNTYNAME"]],
    on="GEOID", validate="many_to_one",
)
# Violent: homicide, rape, robbery, and aggravated assault.
# Property: burglary, larceny-theft, and motor-vehicle theft.
crime = pd.read_csv(
    DOJ_CRIME_PATH,
    usecols=[
        "Year", "County", "Violent_sum", "Property_sum",
        "ViolentClr_sum", "PropertyClr_sum",
    ],
)
crime = crime.loc[crime["Year"].isin(SAFETY_YEARS)].copy()
crime["CountyKey"] = (
    crime["County"].str.removesuffix(" County").str.casefold()
)
county_crime = (
    crime.groupby(["Year", "CountyKey"], as_index=False)[
        ["Violent_sum", "Property_sum", "ViolentClr_sum", "PropertyClr_sum"]
    ].sum().rename(columns={"Year": "SafetyDataYear"})
)
safety_rates = county_crime.merge(
    county_population, on=["SafetyDataYear", "CountyKey"],
    validate="one_to_one",
)
safety_rates["ViolentCrimeRatePer1000"] = (
    1_000 * safety_rates["Violent_sum"] / safety_rates["Population"]
)
safety_rates["PropertyCrimeRatePer1000"] = (
    1_000 * safety_rates["Property_sum"] / safety_rates["Population"]
)
safety_rates["ViolentClearancePct"] = (
    100 * safety_rates["ViolentClr_sum"] / safety_rates["Violent_sum"]
)
safety_rates["PropertyClearancePct"] = (
    100 * safety_rates["PropertyClr_sum"] / safety_rates["Property_sum"]
)
assert len(safety_rates) == len(SAFETY_YEARS) * 58
assert safety_rates[["ViolentCrimeRatePer1000", "PropertyCrimeRatePer1000"]].ge(0).all().all()

In [ ]:
statewide_safety = safety_rates.groupby("SafetyDataYear", as_index=False).agg(
    Population=("Population", "sum"),
    ViolentCrimes=("Violent_sum", "sum"),
    PropertyCrimes=("Property_sum", "sum"),
    ViolentClearances=("ViolentClr_sum", "sum"),
    PropertyClearances=("PropertyClr_sum", "sum"),
)
statewide_safety["ViolentCrimeRatePer1000"] = (
    1_000 * statewide_safety["ViolentCrimes"] / statewide_safety["Population"]
)
statewide_safety["PropertyCrimeRatePer1000"] = (
    1_000 * statewide_safety["PropertyCrimes"] / statewide_safety["Population"]
)
statewide_safety["ViolentClearancePct"] = (
    100 * statewide_safety["ViolentClearances"] / statewide_safety["ViolentCrimes"]
)
statewide_safety["PropertyClearancePct"] = (
    100 * statewide_safety["PropertyClearances"] / statewide_safety["PropertyCrimes"]
)

### Crime-rate features by county

These maps show the two safety measurements used by the models. Rates per 1,000 residents are more comparable than raw counts because California counties have very different populations.

In [ ]:
county_shapes = (
    population_map.assign(CountyGEOID=population_map["GEOID"].str[:5])
    .dissolve(by="CountyGEOID", as_index=False)[["CountyGEOID", "geometry"]]
)
latest_rates = safety_rates.loc[
    safety_rates["SafetyDataYear"].eq(max(SAFETY_YEARS)),
    ["GEOID", "ViolentCrimeRatePer1000", "PropertyCrimeRatePer1000"],
]
county_crime_map = county_shapes.merge(
    latest_rates, left_on="CountyGEOID", right_on="GEOID",
    how="left", validate="one_to_one",
)

map_columns = ["ViolentCrimeRatePer1000", "PropertyCrimeRatePer1000"]
map_titles = ["Violent crime rate", "Property crime rate"]
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for axis, column, title in zip(axes, map_columns, map_titles):
    county_crime_map.plot(
        column=column, cmap="OrRd", edgecolor="white", linewidth=0.3,
        legend=True, legend_kwds={"label": "Incidents per 1,000 residents"},
        ax=axis,
    )
    axis.set_title(title)
    axis.set_axis_off()
fig.suptitle("California county reported-crime rates — 2024")
fig.tight_layout()
plt.show()

### What reported crime and clearance measurements mean

- **Reported crime rate:** offenses reported to law enforcement divided by population, shown per 1,000 residents. A higher rate means more reported offenses relative to population; it is not an individual's probability of becoming a victim.
- **Clearance percentage:** clearances recorded in that year divided by offenses reported in that year. A clearance generally means an arrest or an exceptional clearance—not necessarily a conviction. Because a clearance can concern an earlier offense, this is not a same-year solve rate.

The years are DOJ crime-data years, not property-sale years. Only the two reported-crime rates are model features; clearance percentages are shown for context. Red bars represent violent crime and blue bars represent property crime.

In [ ]:
crime_years = statewide_safety["SafetyDataYear"].astype(str)
x = np.arange(len(crime_years))
width = 0.36
crime_types = [
    ("Violent", "Violent crime", "#B2182B"),
    ("Property", "Property crime", "#2166AC"),
]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

panels = [
    ("CrimeRatePer1000", "Reported incidents per 1,000 residents", "Reported crime rate"),
    ("ClearancePct", "Clearances per 100 reported incidents", "Recorded clearance percentage"),
]
for axis, (suffix, ylabel, title) in zip(axes, panels):
    for offset, (key, label, color) in zip([-width / 2, width / 2], crime_types):
        bars = axis.bar(
            x + offset, statewide_safety[f"{key}{suffix}"], width,
            label=label, color=color, edgecolor="white",
        )
        axis.bar_label(bars, fmt="%.1f", padding=2, fontsize=8)
    axis.set(title=title, ylabel=ylabel, xticks=x, xticklabels=crime_years)
    axis.grid(axis="y", alpha=0.25)
    axis.spines[["top", "right"]].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.suptitle("California reported crimes and clearances, 2022–2024", y=1.04)
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False,
           bbox_to_anchor=(0.5, 0.98))
fig.tight_layout(rect=(0, 0, 1, 0.86))
plt.show()

## 2. Load Notebook 04 outputs

Notebook 04 saves the prepared 33-feature splits, feature schema, and baseline results. Load those files here instead of repeating its preprocessing and model training. **Run Notebook 04 first.**

In [ ]:
# Locate the reusable files created by Notebook 04.
split_paths = {
    "Training": PROCESSED_DIR / "notebook04_training_features.csv.gz",
    "Validation": PROCESSED_DIR / "notebook04_validation_features.csv.gz",
    "Test": PROCESSED_DIR / "notebook04_test_features.csv.gz",
}
schema_path = PROCESSED_DIR / "notebook04_feature_schema.json"
results_path = PROCESSED_DIR / "notebook04_model_results.csv"
required_paths = list(split_paths.values()) + [schema_path, results_path]
missing_files = [str(path) for path in required_paths if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Run Notebook 04 first. Missing: {missing_files}")

In [ ]:
# Reuse Notebook 04's feature names and preprocessing groups.
with schema_path.open() as file:
    schema = json.load(file)

TARGET = schema["target"]
AREA_FEATURES = schema["area"]
PROPERTY_COUNT_FEATURES = schema["property_counts"]
COORDINATE_FEATURES = schema["coordinates"]
AGE_FEATURES = schema["age"]
TEMPORAL_FEATURES = schema["temporal"]
BINARY_FEATURES = schema["binary"]
MISSING_INDICATOR_FEATURES = schema["missing_indicators"]
BASE_CATEGORICAL_FEATURES = schema["categorical"]
HOUSING_FEATURES = schema["model_features"]
assert len(HOUSING_FEATURES) == 33

In [ ]:
# Load the prepared splits and the three baseline model results.
splits = {
    name: pd.read_csv(path, low_memory=False)
    for name, path in split_paths.items()
}
for data in splits.values():
    data[BASE_CATEGORICAL_FEATURES] = (
        data[BASE_CATEGORICAL_FEATURES].astype("string").fillna("Unknown")
    )
notebook04_results = pd.read_csv(results_path)

In [ ]:
# Name the feature columns introduced in Notebook 05.
RATIO_FEATURES = [
    "BathroomsPerBedroom", "LivingAreaPerBedroom", "LotToLivingAreaRatio",
]
SQUARED_AGE_FEATURES = ["PropertyAgeSquared"]
ADDITIONAL_PROPERTY_FEATURES = RATIO_FEATURES + SQUARED_AGE_FEATURES
SAFETY_FEATURES = [
    "ViolentCrimeRatePer1000", "PropertyCrimeRatePer1000",
    "PopulationDensity",
]
DISTRICT_FEATURES = [
    "ElementaryDistrictCode", "HighDistrictCode", "UnifiedDistrictCode",
]
DISTRICT_NAME_COLUMNS = [
    "ElementaryDistrictName", "HighDistrictName", "UnifiedDistrictName",
]

In [ ]:
# Confirm the reused chronological splits.
split_summary = pd.DataFrame([
    [name, data["SourceMonth"].min(), data["SourceMonth"].max(), len(data)]
    for name, data in splits.items()
], columns=["Split", "First month", "Last month", "Rows"]).set_index("Split")
display(split_summary)

## 3. Add safety rates and population density

Each valid property coordinate is spatially joined to a 2024 Census tract to obtain `PopulationDensity`. County names and a two-year lag connect each sale to `ViolentCrimeRatePer1000` and `PropertyCrimeRatePer1000`: 2024 sales use 2022 rates, 2025 sales use 2023 rates, and 2026 sales use 2024 rates. Missing matches are retained and handled by development-only median imputation in the model pipeline.

In [ ]:
# Convert valid property coordinates into map points while retaining row IDs.
def build_property_points(data):
    result = data.reset_index(drop=True).copy()
    result["_PropertyRowID"] = np.arange(len(result))
    valid = result[COORDINATE_FEATURES].notna().all(axis=1)
    coordinates = result.loc[valid, ["_PropertyRowID", "Longitude", "Latitude"]]
    points = gpd.GeoDataFrame(
        coordinates[["_PropertyRowID"]].copy(),
        geometry=gpd.points_from_xy(
            coordinates["Longitude"], coordinates["Latitude"]
        ),
        crs="EPSG:4326",
    )
    return result, valid, points

In [ ]:
# Standardize county names so CRMLS, DOJ, and Census records match.
def normalize_county_name(values):
    return (
        values.astype("string").str.strip()
        .str.replace(r"\s+County$", "", case=False, regex=True)
        .str.casefold()
    )

# Add two-year-lagged crime rates and tract population density to each sale.
def attach_safety_measurements(data):
    result = data.copy()
    sale_year = pd.to_numeric(
        result["SourceMonth"].astype("string").str.extract(r"(\d{4})", expand=False),
        errors="coerce",
    )
    result["SafetyDataYear"] = sale_year - 2
    result["CountyKey"] = normalize_county_name(result["CountyOrParish"])

    rate_columns = [
        "SafetyDataYear", "CountyKey",
        "ViolentCrimeRatePer1000", "PropertyCrimeRatePer1000",
    ]
    result = result.merge(
        safety_rates[rate_columns],
        on=["SafetyDataYear", "CountyKey"],
        how="left", validate="many_to_one", sort=False,
    )

    result, valid_coordinates, points = build_property_points(result)
    tract_columns = ["GEOID", "PopulationDensity", "geometry"]
    tract_matches = gpd.sjoin(
        points.to_crs(population_map.crs),
        population_map[tract_columns],
        how="left", predicate="within",
    )
    assert not tract_matches.index.duplicated().any()
    result["PopulationDensity"] = np.nan
    result.loc[tract_matches.index, "PopulationDensity"] = (
        tract_matches["PopulationDensity"].to_numpy()
    )

    crime_matched = result[[
        "ViolentCrimeRatePer1000", "PropertyCrimeRatePer1000"
    ]].notna().all(axis=1)
    diagnostics = {
        "Crime-rate match %": crime_matched.mean() * 100,
        "Population-density match %": result["PopulationDensity"].notna().mean() * 100,
    }
    return result.drop(columns="_PropertyRowID"), diagnostics

In [ ]:
# Apply the safety joins and confirm that records found matching source data.
join_coverage = {}
for split_name, data in splits.items():
    splits[split_name], join_coverage[split_name] = attach_safety_measurements(data)

join_coverage = pd.DataFrame(join_coverage).T.rename_axis("Split")
display(join_coverage.style.format("{:.1f}%"))

## 4. Add school-district identifiers

Each valid original longitude/latitude pair becomes a point. A point-in-polygon join determines which district boundaries contain it. Coordinate values imputed later by the modeling pipeline are **not** used for district assignment because a ZIP or county median could assign the wrong district.

Elementary, high, and unified matches are pivoted into separate columns. This prevents the expected elementary/high overlap from multiplying property rows. Properties with invalid coordinates or no polygon match receive `Unknown`.

In [ ]:
DISTRICT_TYPE_TO_COLUMNS = {
    "Elementary": ("ElementaryDistrictCode", "ElementaryDistrictName"),
    "High": ("HighDistrictCode", "HighDistrictName"),
    "Unified": ("UnifiedDistrictCode", "UnifiedDistrictName"),
}

In [ ]:
# Match each property point to the district polygons that contain it.
def match_district_boundaries(points):
    columns = ["CDCode", "DistrictName", "DistrictType", "geometry"]
    matches = gpd.sjoin(
        points, districts[columns], how="left", predicate="within"
    ).drop(columns=["geometry", "index_right"])
    matched = matches.dropna(subset=["DistrictType"]).copy()
    overlap = matched.duplicated(
        ["_PropertyRowID", "DistrictType"], keep=False
    )
    overlap_rows = int(matched.loc[overlap, "_PropertyRowID"].nunique())
    matched = (
        matched.sort_values(["_PropertyRowID", "DistrictType", "CDCode"])
        .drop_duplicates(["_PropertyRowID", "DistrictType"])
    )
    return matched, overlap_rows

In [ ]:
# Place elementary, high-school, and unified matches in separate columns.
def map_district_columns(result, matched):
    for district_type, (code_column, name_column) in DISTRICT_TYPE_TO_COLUMNS.items():
        subset = matched.loc[
            matched["DistrictType"].eq(district_type),
            ["_PropertyRowID", "CDCode", "DistrictName"],
        ].set_index("_PropertyRowID")
        result[code_column] = result["_PropertyRowID"].map(subset["CDCode"])
        result[name_column] = result["_PropertyRowID"].map(subset["DistrictName"])

    columns = DISTRICT_FEATURES + DISTRICT_NAME_COLUMNS
    result[columns] = result[columns].astype("string").fillna("Unknown")
    return result

In [ ]:
# Label the combination of district systems serving each property.
def district_service_pattern(data):
    conditions = [
        data["UnifiedDistrictCode"].ne("Unknown"),
        data["ElementaryDistrictCode"].ne("Unknown")
        & data["HighDistrictCode"].ne("Unknown"),
        data["ElementaryDistrictCode"].ne("Unknown"),
        data["HighDistrictCode"].ne("Unknown"),
    ]
    choices = [
        "Unified", "Elementary + High", "Elementary only", "High only",
    ]
    return np.select(conditions, choices, default="Unmatched")

In [ ]:
# Summarize coordinate validity, district coverage, and overlaps.
def district_diagnostics(result, valid_coordinates, overlap_rows):
    matched = result[DISTRICT_FEATURES].ne("Unknown")
    return {
        "Rows": len(result),
        "Valid coordinate %": valid_coordinates.mean() * 100,
        "Elementary matched %": matched["ElementaryDistrictCode"].mean() * 100,
        "High matched %": matched["HighDistrictCode"].mean() * 100,
        "Unified matched %": matched["UnifiedDistrictCode"].mean() * 100,
        "Any district matched %": matched.any(axis=1).mean() * 100,
        "Same-type overlap rows": overlap_rows,
    }

In [ ]:
# Run the district join without adding or dropping property rows.
def attach_school_districts(data):
    result, valid, points = build_property_points(data)
    matched, overlap_rows = match_district_boundaries(points)
    result = map_district_columns(result, matched)
    result["DistrictServicePattern"] = district_service_pattern(result)
    diagnostics = district_diagnostics(result, valid, overlap_rows)

    assert len(result) == len(data) and result["_PropertyRowID"].is_unique
    return result.drop(columns="_PropertyRowID"), diagnostics

In [ ]:
coverage_rows = []
for name, data in splits.items():
    splits[name], diagnostics = attach_school_districts(data)
    coverage_rows.append({"Split": name, **diagnostics})

district_coverage = pd.DataFrame(coverage_rows).set_index("Split")
percent_formats = {
    column: "{:.2f}%"
    for column in district_coverage.columns if column.endswith("%")
}
display(district_coverage.style.format(percent_formats))

In [ ]:
# Verify that district categories look plausible and inspect readable assignments.
service_patterns = pd.DataFrame({
    name: data["DistrictServicePattern"].value_counts()
    for name, data in splits.items()
}).fillna(0).astype(int)
display(service_patterns)

sample_columns = [
    "City", "PostalCode", "Latitude", "Longitude",
    "ElementaryDistrictName", "HighDistrictName", "UnifiedDistrictName",
]
display(
    splits["Training"].loc[
        splits["Training"][DISTRICT_FEATURES].ne("Unknown").any(axis=1), sample_columns
    ].sample(10, random_state=RANDOM_STATE)
)

## 5. Add four property measurements

This final stage adds bathrooms per bedroom, living area per bedroom, lot-to-living-area ratio, and property age squared. Zero denominators become missing instead of infinite. The original columns remain available, so models can use both absolute size and relative composition.

`AmenityCount` remains excluded because it would exactly repeat information already represented by the five individual amenity indicators.

In [ ]:
# Calculate the three property ratios and squared property age.
def add_additional_property_measurements(data):
    out = data.copy()
    bedrooms = out["BedroomsTotal"].replace(0, np.nan)
    living_area = out["LivingArea"].replace(0, np.nan)
    out["BathroomsPerBedroom"] = out["BathroomsTotalInteger"] / bedrooms
    out["LivingAreaPerBedroom"] = out["LivingArea"] / bedrooms
    out["LotToLivingAreaRatio"] = out["LotSizeSquareFeet"] / living_area
    out["PropertyAgeSquared"] = out["PropertyAgeAtSale"].pow(2)
    out[ADDITIONAL_PROPERTY_FEATURES] = out[ADDITIONAL_PROPERTY_FEATURES].replace(
        [np.inf, -np.inf], np.nan
    )
    return out

splits = {
    name: add_additional_property_measurements(data)
    for name, data in splits.items()
}

## 6. Define the four Core ideas comparisons

Each feature set retains every measurement from the preceding set. All models use identical rows and fixed hyperparameters.

### Feature set 1 — Notebook 04 housing baseline (33 columns)

Living area, lot size, bedrooms, bathrooms, garage and parking counts, coordinates, property age, sale-month terms, five individual amenities, 14 missing-value flags, city, county, and ZIP code.

### Feature set 2 — Add safety rates and population density (36 columns)

Add `ViolentCrimeRatePer1000`, `PropertyCrimeRatePer1000`, and `PopulationDensity`.

### Feature set 3 — Add school-district identifiers (39 columns)

Add `ElementaryDistrictCode`, `HighDistrictCode`, and `UnifiedDistrictCode`. These identify district geography, not school quality.

### Feature set 4 — Add four property measurements (43 columns)

Add `BathroomsPerBedroom`, `LivingAreaPerBedroom`, `LotToLivingAreaRatio`, and `PropertyAgeSquared`. The original component columns remain in the model.

In [ ]:
FEATURE_SETS = {
    # 1. Verified Notebook 04 baseline
    "Notebook 04 housing baseline": HOUSING_FEATURES,
    # 2. Add safety and population context
    "Housing + safety rates + population density": (
        HOUSING_FEATURES + SAFETY_FEATURES
    ),
    # 3. Add district geography
    "Housing + safety + district identifiers": (
        HOUSING_FEATURES + SAFETY_FEATURES + DISTRICT_FEATURES
    ),
    # 4. Add the four property measurements
    "Housing + safety + districts + ratios + age squared": (
        HOUSING_FEATURES + SAFETY_FEATURES + DISTRICT_FEATURES
        + ADDITIONAL_PROPERTY_FEATURES
    ),
}
feature_set_summary = pd.DataFrame([
    {
        "Feature set": name,
        "Source columns": len(features),
        "Safety rates + population density": all(f in features for f in SAFETY_FEATURES),
        "District identifiers": all(f in features for f in DISTRICT_FEATURES),
        "Ratios + age squared": all(f in features for f in ADDITIONAL_PROPERTY_FEATURES),
    }
    for name, features in FEATURE_SETS.items()
]).set_index("Feature set")

for name, features in FEATURE_SETS.items():
    assert len(features) == len(set(features)), f"Duplicate feature in {name}"
    assert TARGET not in features
    for split_name, data in splits.items():
        missing = sorted(set(features) - set(data.columns))
        assert not missing, f"{split_name}/{name} missing {missing}"
display(feature_set_summary)

## 7. Build preprocessing for the added feature sets

All learned transformations are fit only on development rows. Safety rates, population density, and property ratios use median imputation, `log1p`, and scaling. District codes are categorical identifiers and use the same cross-fitted target encoder as city, county, and ZIP code.

In [ ]:
# Fill coordinates from ZIP, county, then statewide development-data medians.
class GeographicCoordinateImputer(BaseEstimator, TransformerMixin):
    # Learn the coordinate medians from development rows only.
    def fit(self, X, y=None):
        self.zip_medians_ = X.groupby("PostalCode")[COORDINATE_FEATURES].median()
        self.county_medians_ = X.groupby("CountyOrParish")[COORDINATE_FEATURES].median()
        self.global_medians_ = X[COORDINATE_FEATURES].median()
        assert self.global_medians_.notna().all()
        return self

    # Apply the learned medians without changing the number of rows.
    def transform(self, X):
        result = X.copy()
        for column in COORDINATE_FEATURES:
            result[column] = result[column].fillna(
                result["PostalCode"].map(self.zip_medians_[column])
            )
            result[column] = result[column].fillna(
                result["CountyOrParish"].map(self.county_medians_[column])
            )
            result[column] = result[column].fillna(self.global_medians_[column])
        return result

In [ ]:
# Build numeric imputation, optional log transformation, and scaling.
def numeric_pipeline(*, strategy="median", log_transform=False):
    steps = [("imputer", SimpleImputer(strategy=strategy))]
    if log_transform:
        steps.append(("log1p", FunctionTransformer(
            np.log1p, feature_names_out="one-to-one"
        )))
    return Pipeline(steps + [("scaler", StandardScaler())])

In [ ]:
# Fill missing categories and apply cross-fitted target encoding.
def categorical_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(
            strategy="constant", fill_value="Unknown"
        )),
        ("encoder", TargetEncoder(
            target_type="continuous", cv=5, shuffle=True,
            random_state=RANDOM_STATE,
        )),
    ])

In [ ]:
# Select the transformations needed by one cumulative feature set.
def build_preprocessor(features):
    has_safety = set(SAFETY_FEATURES).issubset(features)
    has_ratios = set(RATIO_FEATURES).issubset(features)
    has_districts = set(DISTRICT_FEATURES).issubset(features)
    transformers = [
        ("area", numeric_pipeline(log_transform=True), AREA_FEATURES),
        ("counts", numeric_pipeline(strategy="most_frequent"), PROPERTY_COUNT_FEATURES),
        ("coordinates", StandardScaler(), COORDINATE_FEATURES),
        ("age", numeric_pipeline(), AGE_FEATURES),
        ("temporal", "passthrough", TEMPORAL_FEATURES),
        ("binary", SimpleImputer(strategy="most_frequent"), BINARY_FEATURES),
        ("missing", "passthrough", MISSING_INDICATOR_FEATURES),
    ]
    if has_safety:
        transformers.append(
            ("safety", numeric_pipeline(log_transform=True), SAFETY_FEATURES)
        )
    if has_ratios:
        transformers += [
            ("ratios", numeric_pipeline(log_transform=True), RATIO_FEATURES),
            ("age_squared", numeric_pipeline(), SQUARED_AGE_FEATURES),
        ]
    categorical = BASE_CATEGORICAL_FEATURES + (
        DISTRICT_FEATURES if has_districts else []
    )
    transformers.append(("categorical", categorical_pipeline(), categorical))
    return ColumnTransformer(transformers, verbose_feature_names_out=False)

In [ ]:
# Combine coordinate filling, preprocessing, and a fresh model copy.
def build_model_pipeline(regressor, features):
    return Pipeline([
        ("coordinate_imputation", GeographicCoordinateImputer()),
        ("preprocessor", build_preprocessor(features)),
        ("regressor", clone(regressor)),
    ])

## 8. Train the three models on the added feature sets

Training and March validation rows form the development data. April 2026 remains the common test month. Notebook 04 supplies the housing-baseline results; this notebook trains only feature sets 2–4 with the same model settings.

In [ ]:
development = pd.concat([splits["Training"], splits["Validation"]], ignore_index=True)
test = splits["Test"].copy()
y_development_log = np.log1p(development[TARGET])
y_test = test[TARGET]

models = {
    "Linear Regression": LinearRegression(n_jobs=-1),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=18, min_samples_leaf=10, random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=30, max_depth=18, min_samples_leaf=10,
        max_features=0.7, max_samples=0.7,
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
}
print(f"Development rows: {len(development):,}")
print(f"April test rows: {len(test):,}")

In [ ]:
# Fit one new model/feature-set pair and return April 2026 metrics.
def evaluate_configuration(model_name, regressor, feature_set_name, features):
    print(f"Fitting {model_name} with {feature_set_name}...")
    pipeline = build_model_pipeline(regressor, features)
    pipeline.fit(development[features], y_development_log)
    predictions = np.expm1(pipeline.predict(test[features]))
    metrics = {
        "Model": model_name,
        "Feature set": feature_set_name,
        "Source columns": len(features),
        "Test R²": r2_score(y_test, predictions),
        "Test MAE": mean_absolute_error(y_test, predictions),
        "Test RMSE": mean_squared_error(y_test, predictions) ** 0.5,
    }
    del pipeline, predictions
    gc.collect()
    return metrics

In [ ]:
added_results = pd.DataFrame([
    evaluate_configuration(model_name, regressor, feature_set_name, features)
    for feature_set_name, features in list(FEATURE_SETS.items())[1:]
    for model_name, regressor in models.items()
])

baseline_results = notebook04_results[[
    "Model", "Test R²", "Test MAE", "Test RMSE",
]].copy()
baseline_results["Feature set"] = "Notebook 04 housing baseline"
baseline_results["Source columns"] = len(HOUSING_FEATURES)
results = pd.concat([baseline_results, added_results], ignore_index=True)

In [ ]:
baseline_r2 = (
    results.loc[
        results["Feature set"].eq("Notebook 04 housing baseline"),
        ["Model", "Test R²"],
    ]
    .set_index("Model")["Test R²"]
)
results["ΔR² vs housing baseline"] = (
    results["Test R²"] - results["Model"].map(baseline_r2)
)

## 9. Compare the Core ideas feature sets

Positive ΔR² means a feature set improved the same model relative to the verified Notebook 04 housing baseline. MAE and RMSE are reported in dollars; lower values are better.

The grouped bar chart compares absolute April 2026 test R² across models and feature sets. Its y-axis is deliberately zoomed to R² = 0.40–0.90, with the truncated baseline stated in the title and axis label. Exact R² values are printed above the bars.

In [ ]:
feature_set_order = list(FEATURE_SETS)
model_order = list(models)
results["Feature set"] = pd.Categorical(
    results["Feature set"], categories=feature_set_order, ordered=True
)
results["Model"] = pd.Categorical(
    results["Model"], categories=model_order, ordered=True
)
results = results.sort_values(["Feature set", "Model"]).reset_index(drop=True)
display(results.style.format({
    "Test R²": "{:.4f}",
    "Test MAE": "${:,.0f}",
    "Test RMSE": "${:,.0f}",
    "ΔR² vs housing baseline": "{:+.4f}",
}))

In [ ]:
# Reshape once for the grouped bar chart; the detailed table above is the only table shown.
r2_comparison = results.pivot(
    index="Model", columns="Feature set", values="Test R²"
)

In [ ]:
short_labels = [
    "Housing\nbaseline",
    "Add crime rates\n+ population density",
    "Add district\nidentifiers",
    "Add 3 ratios\n+ age squared",
]
colors = {
    "Linear Regression": "#7f8c8d",
    "Decision Tree": "#f39c12",
    "Random Forest": "#2e86de",
}
bar_width = 0.24
x = np.arange(len(feature_set_order))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for offset, model_name in zip([-bar_width, 0, bar_width], model_order):
    values = r2_comparison.loc[model_name, feature_set_order].to_numpy(float)
    bars = ax.bar(
        x + offset, values, width=bar_width,
        label=model_name, color=colors[model_name],
    )
    ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=9)

ax.set(
    ylabel="Test R² (axis starts at 0.40)",
    xticks=x, xticklabels=short_labels,
    ylim=(0.40, 0.90), yticks=np.arange(0.40, 0.91, 0.10),
)
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(
    "April 2026 test R² by model and feature set\n"
    "(zoomed; baseline = 0.40)", y=0.99,
)
fig.legend(*ax.get_legend_handles_labels(), frameon=False, ncol=3,
           loc="upper center", bbox_to_anchor=(0.5, 0.90))
fig.tight_layout(rect=(0, 0, 1, 0.82))
plt.show()

In [ ]:
best_rows = (
    results.sort_values("Test R²", ascending=False)
    .groupby("Model", observed=True, as_index=False)
    .first()[["Model", "Feature set", "Test R²", "ΔR² vs housing baseline"]]
)
display(best_rows.style.format({
    "Test R²": "{:.4f}",
    "ΔR² vs housing baseline": "{:+.4f}",
}).set_caption("Best tested feature set for each model"))

## 10. Findings

- **Notebook 04 baseline:** reused directly rather than retrained here.
- **Safety and population:** improved all three models; Random Forest rose from 0.8755 to 0.8790 R².
- **District identifiers:** produced the best Decision Tree (0.8541) and Random Forest (0.8792) results.
- **Four property measurements:** did not improve the best result.

Best overall: **Random Forest with safety, population density, and district identifiers (R² = 0.8792).**

## 11. Interpretation limits

- District boundaries are from 2024–25, and ACS population values summarize 2020–2024.
- District IDs describe geography, not assigned schools or school quality.
- County reported-crime rates cannot represent neighborhood-level or unreported safety differences.
- Results show predictive association, not causation; April 2026 is a reused project benchmark.